In [12]:
import pandas as pd
import requests
import sys
import io
from io import  StringIO

In [13]:
    url = "https://en.wikipedia.org/wiki/Opinion_polling_for_the_next_United_Kingdom_general_election"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }

In [14]:
response = requests.get(url, headers=headers)
response.raise_for_status()
        
html_content = StringIO(response.text)
        
tables = pd.read_html(html_content, flavor='lxml')

if not tables:
    print("No tables were found on the page.")

In [15]:
def clean_data(df):
    # The html table has a two line header which is not correct for dataframes
    # this seems to prevent normal operations on the data frame, so save it as a csv, reload ir and delete the first row
    df.to_csv('temp.csv', index=False)
    df1 = pd.read_csv("temp.csv")
    df1 = df1.tail(-1)
    
    # To remove 'note' rows change a col to numeric. Notes will be strings which convert to None
    # then drop the rows with None thus removing the notes
    df1['Sample size'] = pd.to_numeric(df1['Sample size'],errors='coerce')
    df1 = df1.dropna(subset=['Sample size']) # remove 'note' rows
    
    # Rename abbreviations
    df1 = df1.rename(columns={
        'Con': 'Conservative',
        'Lab': 'Labour',
        'LD': 'Liberal Democrats',
        'Grn': 'Green',
        'Ref': 'Reform',
        'PC': 'Plaid Cymru'
    }, errors="raise")
    
    # Convert party columns to numeric. Replace '-' with None, replace '%' 
    party_cols = ['Conservative', 'Labour', 'Liberal Democrats', 'Green', 'Reform', 'SNP', 'Plaid Cymru', 'Others']
    df1[party_cols]= (   
        df1[party_cols].replace(r"–", None, regex=True)
    )
    df1[party_cols] = (
        df1[party_cols].replace(r"%","", regex=True)
    )
    
    # Remove the refs from the Pollsters name, e.g. 'Ipsos[2]' becomes simply 'Ipsos'
    df1['Pollster'] = df1['Pollster'].str.replace(r'\[[A-Za-z0-9]+\]', '', regex=True)
    df1['Pollster'] = df1['Pollster'].str.replace(r' \(MRP\)', '', regex=True)
    # JL Partners is sometimes styled J.L. Partners
    df1['Pollster'] = df1['Pollster'].str.replace('J.L.', 'JL')
    
    df1['Pollster'] = df1['Pollster'].str.strip()
    return df1

In [16]:
def parse_start_date(s, year):
    s = str(s).replace("–", "-")   # normalize dash
    #print(#s)
    
    if s.find("-")>-1: 
        start = s.split("-")[1].strip()
        #print("startwith-",s, start)
    else:
        start = s.strip()
        #print("startwithout-",s, start)
    
    d = pd.to_datetime(f"{start} {year}", dayfirst=True)
    #print(d)
    return d





In [17]:
table = 1
year = "2026"

df = tables[table]
#
df.head()

df1 = clean_data(df)
df1["Date"] = df1["Date(s) conducted"].apply(parse_start_date, year=year)

df1.to_csv(f"{year}.csv", index=False)

In [18]:
df1.head()

,Date(s) conducted,Pollster,Client,Area,Sample size,Labour,Conservative,Reform,Liberal Democrats,Green,SNP,Plaid Cymru,Others,Lead,Date
1,22–23 Apr,Find Out Now,NaN,GB,2656.0,15,16,25,11,21,3,2,7,4,2026-04-23
2,19–20 Apr,YouGov,The Times/Sky News,GB,2472.0,16,17,27,14,17,3,1,5 RB 3 YP 0 Other 2,10,2026-04-20
3,17–20 Apr,More in Common,NaN,GB,2235.0,20,22,27,11,12,3,1,5,5,2026-04-20
4,15–17 Apr,Opinium,The Observer,GB,2014.0,22,17,26,11,15,3,1,5,4,2026-04-17
5,15–16 Apr,Find Out Now,Restore Britain,GB,2284.0,17,18,21,11,18,3,1,13 RB 9 YP 1 Other 3,3,2026-04-16


In [19]:
table = 2
year = "2025"

df = tables[table]

df2 = clean_data(df)
df2["Date"] = df2["Date(s) conducted"].apply(parse_start_date, year=year)

df2.to_csv(f"{year}.csv", index=False)
df2.head()

,Date(s) conducted,Pollster,Client,Area,Sample size,Labour,Conservative,Reform,Liberal Democrats,Green,SNP,Plaid Cymru,Others,Lead,Date
1,31 Dec,Find Out Now,NaN,GB,2930.0,15,17,31,12,17,3,2,4,14,2025-12-31
2,24 Dec,Find Out Now,NaN,GB,2879.0,14,18,30,12,17,3,1,4,12,2025-12-24
3,19–23 Dec,More in Common,NaN,GB,2026.0,21,22,28,13,9,3,1,3,6,2025-12-23
4,21–22 Dec,YouGov,NaN,GB,2266.0,20,19,25,15,15,3,1,2,5,2025-12-22
5,16–18 Dec,Deltapoll,The Mirror,GB,1997.0,20,19,30,14,10,3,1,2,10,2025-12-18


In [20]:
dfAll = pd.concat([df1,df2])
dfAll.to_csv(f"all.csv", index=False)



In [21]:
dfAll.head()

dfOrig = dfAll.copy()


In [24]:
cols_to_drop = [
    "Date(s) conducted",
    "Sample size",
    "Area",
    "Client",
    "Client Area",
    "Others",
    "Lead"
]
dfAll = dfAll.drop(columns=cols_to_drop, errors="ignore")
dfAll.head()

,Pollster,Labour,Conservative,Reform,Liberal Democrats,Green,SNP,Plaid Cymru,Date
1,Find Out Now,15,16,25,11,21,3,2,2026-04-23
2,YouGov,16,17,27,14,17,3,1,2026-04-20
3,More in Common,20,22,27,11,12,3,1,2026-04-20
4,Opinium,22,17,26,11,15,3,1,2026-04-17
5,Find Out Now,17,18,21,11,18,3,1,2026-04-16


In [23]:
# 1. Get the latest date
latest_date = dfAll['Date'].max()

# 2. Calculate one year before
one_year_before = latest_date - pd.DateOffset(years=1)
six_months_before = latest_date - pd.DateOffset(months=6)
three_months_before = latest_date - pd.DateOffset(months=3)
one_month_before = latest_date - pd.DateOffset(months=1)

# 3. Filter dataframe for the last year
#ytod = dfAll[dfAll['Date'] >= one_year_before]
ytod = dfAll[(dfAll['Date'] >= one_year_before) & (dfAll['Date'] <= latest_date)]

print("Latest date:", latest_date)
print("One year before:", one_year_before)
print("Filtered min:", ytod['Date'].min())
print("Filtered max:", ytod['Date'].max())
print(ytod[-1:]['Date'])



Latest date: 2026-04-23 00:00:00
One year before: 2025-04-23 00:00:00
Filtered min: 2025-04-23 00:00:00
Filtered max: 2026-04-23 00:00:00
199   2025-04-23
Name: Date, dtype: datetime64[us]


In [25]:
party_cols = ['Labour', 'Conservative', 'Reform', 'Liberal Democrats', 'Green', 'SNP', 'Plaid Cymru']

df_long = dfAll.melt(
    id_vars=['Pollster', 'Date'],
    value_vars=party_cols,
    var_name='Party',
    value_name='Percent'
)

df_long['Percent'] = pd.to_numeric(df_long['Percent'], errors='coerce')
df_long.head()

,Pollster,Date,Party,Percent
0,Find Out Now,2026-04-23,Labour,15.0
1,YouGov,2026-04-20,Labour,16.0
2,More in Common,2026-04-20,Labour,20.0
3,Opinium,2026-04-17,Labour,22.0
4,Find Out Now,2026-04-16,Labour,17.0


In [26]:
df_long_12m = df_long[df_long['Date'] >= one_year_before]
df_long_6m = df_long[df_long['Date'] >= six_months_before]
df_long_3m = df_long[df_long['Date'] >= three_months_before]
df_long_1m = df_long[df_long['Date'] >= one_month_before]

In [28]:
import json

# Save each dataframe as JSON
df_long_12m.to_json("12months.json", orient="records", date_format="iso")
df_long_6m.to_json("6months.json", orient="records", date_format="iso")
df_long_3m.to_json("3months.json", orient="records", date_format="iso")
df_long_1m.to_json("1month.json", orient="records", date_format="iso")
dfAll.to_json("all.json", orient="records", date_format="iso")

In [ ]:

ytod = dfAll[(dfAll['Date'] >= one_year_before) & (dfAll['Date'] <= latest_date)]
months6 = dfAll[(dfAll['Date'] >= six_months_before) & (dfAll['Date'] <= latest_date)]
months3 = dfAll[(dfAll['Date'] >= three_months_before) & (dfAll['Date'] <= latest_date)]
months1 = dfAll[(dfAll['Date'] >= one_month_before) & (dfAll['Date'] <= latest_date)]
ytod.to_csv(f"12months.csv", index=False)
months6.to_csv(f"6months.csv", index=False)
months3.to_csv(f"3months.csv", index=False)
months1.to_csv(f"1month.csv", index=False)

dfAll.to_json(f"all.json", orient="records", date_format="iso")
ytod.to_json(f"12months.json", orient="records", date_format="iso")
months6.to_json(f"6months.json", orient="records", date_format="iso")
months3.to_json(f"3months.json", orient="records", date_format="iso")
months1.to_json(f"1month.json", orient="records", date_format="iso")
